<a href="https://colab.research.google.com/github/Akshayextreme/Fake_news_detection_hackathon/blob/master/Fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
import numpy as np
import pandas as pd
from sklearn import metrics
import transformers
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

In [ ]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

In [ ]:
df = pd.read_csv('Train.csv')

In [ ]:
df.head()

,Labels,Text,Text_Tag
0,1,Says the Annies List political group supports ...,abortion
1,2,When did the decline of coal start? It started...,"energy,history,job-accomplishments"
2,3,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy
3,1,Health care reform legislation is likely to ma...,health-care
4,2,The economic turnaround started at the end of ...,"economy,jobs"


In [ ]:
df.isnull().sum()

Labels      0
Text        0
Text_Tag    2
dtype: int64

In [ ]:
df.Text_Tag.fillna('Not Available', inplace=True)

In [ ]:
df['Text_Tag'] = df.Text_Tag.str.replace(",", " ")

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


In [ ]:
def leng_txt(x):
    in_ids = tokenizer.encode(x, add_special_tokens=True)
    return len(in_ids)

In [ ]:
df['length'] = df.Text.apply(leng_txt)

Token indices sequence length is longer than the specified maximum sequence length for this model (714 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
df.length.quantile(0.95)

42.0

In [ ]:
df.length.median()

22.0

In [ ]:
MAX_LEN = 50
TRAIN_BATCH_SIZE = 64
VALID_BATCH_SIZE = 32
LEARNING_RATE = 1e-05

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe
        self.text = dataframe.Text
        self.targets = dataframe.Labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        text = str(self.text[idx])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=True,
            truncation=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[idx], dtype=torch.long)
        }

In [ ]:
train_size = 0.8
train_dataset=df.sample(frac=train_size,random_state=648)
val_dataset=df.drop(train_dataset.index).reset_index(drop=True)
train_dataset = train_dataset.reset_index(drop=True)


print("FULL Dataset: {}".format(df.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(val_dataset.shape))

training_set = CustomDataset(train_dataset, tokenizer, MAX_LEN)
validation_set = CustomDataset(val_dataset, tokenizer, MAX_LEN)

FULL Dataset: (10240, 4)
TRAIN Dataset: (8192, 4)
TEST Dataset: (2048, 4)


In [ ]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 4
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 4
                }

training_loader = DataLoader(training_set, **train_params)
validation_loader = DataLoader(validation_set, **test_params)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:560: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels = 6,
    output_attentions = False,
    output_hidden_states = False,
)

model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.weight', 'classifier.bias', 'classifier.weight', 'pre_classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

In [ ]:
from transformers import get_linear_schedule_with_warmup
EPOCHS = 30
total_steps = len(training_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0,
                                            num_training_steps = total_steps)

In [ ]:
def validation(epoch):
    model.eval()
    running_loss = 0.0
    running_corrects = 0

    with torch.no_grad():
        for _, data in enumerate(validation_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.long)

            loss, outputs = model(ids, mask, labels=targets,  return_dict=False)
            preds = torch.argmax(outputs, 1)

            running_loss += loss.item()
            running_corrects += torch.sum(preds == targets.data)

        epoch_loss = running_loss / validation_set.__len__()
        epoch_acc = running_corrects.double() / validation_set.__len__()

        print(f'Valid -> Epoch: {epoch}, Loss: {epoch_loss}, Accuracy: {epoch_acc}')
        print('\n ==================================================================== \n')
        return epoch_acc

In [ ]:
def train(epoch):
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for i, data in enumerate(training_loader, 0):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.long)

        loss, outputs = model(ids, mask, labels=targets, return_dict=False)
        preds = torch.argmax(outputs, 1)

        optimizer.zero_grad()

        running_loss += loss.item()
        running_corrects += torch.sum(preds == targets.data)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        scheduler.step()

    epoch_loss = running_loss / training_set.__len__()
    epoch_acc = running_corrects.double() / training_set.__len__()
    print(f'Train -> Epoch: {epoch}, Loss: {epoch_loss}, Accuracy: {epoch_acc}')

In [ ]:
import copy
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0

for epoch in range(EPOCHS):
    train(epoch)
    epoch_acc = validation(epoch)
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        best_model_wts = copy.deepcopy(model.state_dict())

model.load_state_dict(best_model_wts)
print(f'Best Valid Acc: {best_acc}')

Train -> Epoch: 0, Loss: 0.02523192556691356, Accuracy: 0.3240966796875
Valid -> Epoch: 0, Loss: 0.052398275467567146, Accuracy: 0.26318359375


Train -> Epoch: 1, Loss: 0.024401132643106394, Accuracy: 0.36572265625
Valid -> Epoch: 1, Loss: 0.052679625048767775, Accuracy: 0.2705078125


Train -> Epoch: 2, Loss: 0.023277245709323324, Accuracy: 0.4024658203125
Valid -> Epoch: 2, Loss: 0.053145208570640534, Accuracy: 0.27587890625


Train -> Epoch: 3, Loss: 0.022217662670300342, Accuracy: 0.441162109375
Valid -> Epoch: 3, Loss: 0.05421661405125633, Accuracy: 0.265625


Train -> Epoch: 4, Loss: 0.020991119206883013, Accuracy: 0.47900390625
Valid -> Epoch: 4, Loss: 0.05556525313295424, Accuracy: 0.263671875


Train -> Epoch: 5, Loss: 0.019855501581332646, Accuracy: 0.52001953125
Valid -> Epoch: 5, Loss: 0.05709492013556883, Accuracy: 0.26220703125


Train -> Epoch: 6, Loss: 0.018608463760756422, Accuracy: 0.55517578125
Valid -> Epoch: 6, Loss: 0.05901740776607767, Accuracy: 0.25830078125




In [ ]:
model.save_pretrained(r"C:\Users\Meena\Downloads\Fake News Pred")
model.config.save_pretrained(r"C:\Users\Meena\Downloads\Fake News Pred")

### Test

In [ ]:
df_test = pd.read_csv('Test.csv')
df_test['Labels'] = 0

In [ ]:
df_test['Text_Tag'] = df_test.Text_Tag.str.replace(",", " ")

In [ ]:
df_test['length'] = df_test.Text.apply(leng_txt)
df_test.length.quantile(0.95)

41.0

In [ ]:
testing_set = CustomDataset(df_test, tokenizer, MAX_LEN)
TEST_BATCH_SIZE = 32
test_params = {'batch_size': TEST_BATCH_SIZE,
                'shuffle': False,
                'num_workers': 4
                }
testing_loader = DataLoader(testing_set, **test_params)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:560: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [ ]:
proba = torch.nn.Softmax(dim=1)

def test():
    model.eval()
    prediction = []

    with torch.no_grad():
        for i, data in enumerate(testing_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            outputs = model(ids, mask)
            outputs = proba(outputs[0])
            outputs = outputs.detach().cpu().numpy()
            prediction.append(outputs)
        return prediction

In [ ]:
tmp = test()

In [ ]:
submit = pd.DataFrame(np.vstack(tmp))

In [ ]:
submit.to_csv('submit.csv', index=False)

In [ ]:
submit

,0,1,2,3,4,5
0,0.196518,0.424252,0.130565,0.047329,0.151098,0.050239
1,0.096443,0.095540,0.433905,0.230108,0.025010,0.118995
2,0.325665,0.291296,0.135231,0.072313,0.068706,0.106789
3,0.392682,0.273385,0.158150,0.042405,0.098953,0.034424
4,0.179562,0.180745,0.232442,0.143677,0.055144,0.208430
...,...,...,...,...,...,...
1262,0.060289,0.114713,0.120878,0.387473,0.019160,0.297487
1263,0.193930,0.260595,0.152594,0.123361,0.085424,0.184097
1264,0.279242,0.224301,0.163940,0.097174,0.115844,0.119499
1265,0.332718,0.234525,0.129954,0.086224,0.151101,0.065479
